# 05 — Predict & Evaluate Model Accuracy

Load the trained GraphSAGE + WGAN-GP models, run predictions on the held-out
test set, and compute full accuracy metrics to validate the pipeline.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc as sk_auc,
    precision_recall_curve, average_precision_score,
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score,
)

from gan_anomaly import Generator, Encoder, anomaly_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Pipeline integration ──────────────────────────────────────────
_RUN_DIR = os.environ.get("AML_RUN_DIR", "")
DATA_DIR = os.path.join(_RUN_DIR, "data") if _RUN_DIR else "data"
EMBEDDINGS_DIR = os.path.join(_RUN_DIR, "embeddings") if _RUN_DIR else "embeddings"
MODELS_DIR = os.path.join(_RUN_DIR, "models") if _RUN_DIR else "models"
RESULTS_DIR = os.path.join(_RUN_DIR, "results") if _RUN_DIR else "results"

os.makedirs(RESULTS_DIR, exist_ok=True)

## 1. Load Trained Models

In [ ]:
with open(os.path.join(MODELS_DIR, "training_meta.json")) as f:
    meta = json.load(f)

INPUT_DIM = meta["input_dim"]
LATENT_DIM = meta["latent_dim"]

G = Generator(LATENT_DIM, INPUT_DIM, meta["g_hidden"], meta["n_layers"], meta["activation"]).to(device)
E = Encoder(INPUT_DIM, LATENT_DIM, meta["e_hidden"], meta["n_layers"], meta["activation"]).to(device)

G.load_state_dict(torch.load(os.path.join(MODELS_DIR, "generator.pt"), map_location=device, weights_only=True))
E.load_state_dict(torch.load(os.path.join(MODELS_DIR, "encoder.pt"), map_location=device, weights_only=True))
G.eval(); E.eval()

print(f"Loaded Generator + Encoder (input_dim={INPUT_DIM}, latent_dim={LATENT_DIM})")

## 2. Load Train, Test & Eval Data

In [ ]:
X_train = np.load(os.path.join(MODELS_DIR, "X_train.npy"))
X_eval = np.load(os.path.join(MODELS_DIR, "X_eval.npy"))
y_eval = np.load(os.path.join(MODELS_DIR, "y_eval.npy"))

embeddings = np.load(os.path.join(EMBEDDINGS_DIR, "node_embeddings.npy"))
node_ids = np.load(os.path.join(EMBEDDINGS_DIR, "node_ids.npy"), allow_pickle=True)
node_features = pd.read_parquet(os.path.join(DATA_DIR, "node_features.parquet"))

is_sar_all = node_features.set_index("id")["is_sar"].reindex(node_ids).fillna(0).astype(int).values

print(f"Training samples (non-SAR only): {len(X_train):,}")
print(f"Eval samples (test+SAR):         {len(X_eval):,}")
print(f"  - Non-SAR in eval:             {int((y_eval == 0).sum()):,}")
print(f"  - SAR in eval:                 {int((y_eval == 1).sum()):,}")
print(f"All nodes:                       {len(embeddings):,}")

## 3. Predict — Score Training Set (establish baseline)

In [37]:
train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
train_scores = anomaly_score(train_tensor, E, G).cpu().numpy()

print(f"Training scores (clean accounts only):")
print(f"  Mean:   {train_scores.mean():.4f}")
print(f"  Std:    {train_scores.std():.4f}")
print(f"  Median: {np.median(train_scores):.4f}")
print(f"  Min:    {train_scores.min():.4f}")
print(f"  Max:    {train_scores.max():.4f}")

for pct in [90, 95, 99, 99.5]:
    print(f"  P{pct}:  {np.percentile(train_scores, pct):.4f}")

Training scores (clean accounts only):
  Mean:   0.8371
  Std:    4.6008
  Median: 0.3442
  Min:    0.0701
  Max:    302.1164
  P90:  1.3545
  P95:  2.5013
  P99:  6.0661
  P99.5:  8.8785


## 4. Predict — Score Eval Set (test + SAR)

In [38]:
eval_tensor = torch.tensor(X_eval, dtype=torch.float32).to(device)
eval_scores = anomaly_score(eval_tensor, E, G).cpu().numpy()

# Separate scores by class
nonsar_eval_scores = eval_scores[y_eval == 0]
sar_eval_scores = eval_scores[y_eval == 1]

print(f"Non-SAR eval scores:  mean={nonsar_eval_scores.mean():.4f}  std={nonsar_eval_scores.std():.4f}")
print(f"SAR eval scores:      mean={sar_eval_scores.mean():.4f}  std={sar_eval_scores.std():.4f}")
print(f"\nSAR scores are {'HIGHER' if sar_eval_scores.mean() > nonsar_eval_scores.mean() else 'LOWER'} than non-SAR")
print(f"  → Model {'CAN' if sar_eval_scores.mean() > nonsar_eval_scores.mean() else 'CANNOT'} distinguish SAR from clean")

Non-SAR eval scores:  mean=0.7588  std=1.4508
SAR eval scores:      mean=1.5766  std=6.3291

SAR scores are HIGHER than non-SAR
  → Model CAN distinguish SAR from clean


## 5. Set Threshold & Classify

In [39]:
# Try multiple threshold strategies
thresholds = {
    "P95 (train)": np.percentile(train_scores, 95),
    "P99 (train)": np.percentile(train_scores, 99),
    "P99.5 (train)": np.percentile(train_scores, 99.5),
}

print(f"{'Threshold':<20s} {'Value':>8s} {'Accuracy':>10s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s}")
print("-" * 70)

for name, thr in thresholds.items():
    y_pred = (eval_scores > thr).astype(int)
    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)
    print(f"{name:<20s} {thr:>8.4f} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}")

Threshold               Value   Accuracy  Precision     Recall         F1
----------------------------------------------------------------------
P95 (train)            2.5013     0.6716     0.4182     0.0714     0.1220
P99 (train)            6.0661     0.6806     0.5000     0.0248     0.0473
P99.5 (train)          8.8785     0.6830     0.6316     0.0186     0.0362


## 6. Optimal Threshold (Youden's J)

In [40]:
fpr, tpr, roc_thresholds = roc_curve(y_eval, eval_scores)
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
optimal_threshold = roc_thresholds[best_idx]

y_pred_optimal = (eval_scores > optimal_threshold).astype(int)

print(f"Optimal threshold (Youden's J): {optimal_threshold:.4f}")
print(f"  TPR at optimal: {tpr[best_idx]:.4f}")
print(f"  FPR at optimal: {fpr[best_idx]:.4f}")
print(f"  J statistic:    {j_scores[best_idx]:.4f}")
print()
print(f"Accuracy:  {accuracy_score(y_eval, y_pred_optimal):.4f}")
print(f"Precision: {precision_score(y_eval, y_pred_optimal, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_eval, y_pred_optimal, zero_division=0):.4f}")
print(f"F1:        {f1_score(y_eval, y_pred_optimal, zero_division=0):.4f}")

Optimal threshold (Youden's J): 0.3857
  TPR at optimal: 0.6925
  FPR at optimal: 0.4461
  J statistic:    0.2465

Accuracy:  0.5977
Precision: 0.4210
Recall:    0.6910
F1:        0.5232


## 7. Full Classification Report (Optimal Threshold)

In [41]:
print("Classification Report:")
print(classification_report(y_eval, y_pred_optimal, target_names=["Non-SAR", "SAR"]))

cm = confusion_matrix(y_eval, y_pred_optimal)
print("Confusion Matrix:")
print(f"                Predicted")
print(f"                Non-SAR    SAR")
print(f"  Actual Non-SAR  {cm[0][0]:>5d}    {cm[0][1]:>5d}")
print(f"  Actual SAR      {cm[1][0]:>5d}    {cm[1][1]:>5d}")

Classification Report:
              precision    recall  f1-score   support

     Non-SAR       0.79      0.55      0.65      1372
         SAR       0.42      0.69      0.52       644

    accuracy                           0.60      2016
   macro avg       0.61      0.62      0.59      2016
weighted avg       0.67      0.60      0.61      2016

Confusion Matrix:
                Predicted
                Non-SAR    SAR
  Actual Non-SAR    760      612
  Actual SAR        199      445


## 8. ROC AUC

In [ ]:
roc_auc = roc_auc_score(y_eval, eval_scores)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, linewidth=2, label=f"WGAN-GP (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random (AUC = 0.50)")
plt.scatter(fpr[best_idx], tpr[best_idx], color="red", s=100, zorder=5,
            label=f"Optimal (thr={optimal_threshold:.3f})")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve — AML Anomaly Detection")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "roc_curve_eval.png"), dpi=150)
plt.show()

print(f"\nROC AUC: {roc_auc:.4f}")

## 9. Precision-Recall Curve

In [ ]:
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_eval, eval_scores)
ap = average_precision_score(y_eval, eval_scores)

plt.figure(figsize=(7, 6))
plt.plot(recall_curve, precision_curve, linewidth=2, label=f"WGAN-GP (AP = {ap:.4f})")
baseline = y_eval.sum() / len(y_eval)
plt.axhline(y=baseline, color="k", linestyle="--", alpha=0.4, label=f"Baseline ({baseline:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — AML Anomaly Detection")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "pr_curve_eval.png"), dpi=150)
plt.show()

print(f"\nAverage Precision: {ap:.4f}")

## 10. Score Distributions — Train vs Test vs SAR

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: train vs eval non-SAR (should look similar = no overfitting)
ax = axes[0]
ax.hist(train_scores, bins=60, alpha=0.6, label=f"Train non-SAR (n={len(train_scores)})", density=True, color="steelblue")
ax.hist(nonsar_eval_scores, bins=60, alpha=0.6, label=f"Test non-SAR (n={len(nonsar_eval_scores)})", density=True, color="orange")
ax.set_xlabel("Anomaly Score")
ax.set_ylabel("Density")
ax.set_title("Train vs Test (non-SAR) — Overfit Check")
ax.legend()
ax.grid(True, alpha=0.3)

# Right: test non-SAR vs SAR (should be separable)
ax = axes[1]
ax.hist(nonsar_eval_scores, bins=60, alpha=0.6, label=f"Non-SAR (n={len(nonsar_eval_scores)})", density=True, color="steelblue")
ax.hist(sar_eval_scores, bins=60, alpha=0.6, label=f"SAR (n={len(sar_eval_scores)})", density=True, color="crimson")
ax.axvline(optimal_threshold, color="black", linestyle="--", label=f"Threshold={optimal_threshold:.3f}")
ax.set_xlabel("Anomaly Score")
ax.set_ylabel("Density")
ax.set_title("Non-SAR vs SAR — Separation")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "score_comparison.png"), dpi=150)
plt.show()

## 11. Predict on All Nodes — Final Output

In [45]:
all_tensor = torch.tensor(embeddings, dtype=torch.float32).to(device)
all_scores = anomaly_score(all_tensor, E, G).cpu().numpy()

# ── Free GPU immediately — rest of notebook is CPU only ──
del all_tensor, train_tensor, eval_tensor, G, E
torch.cuda.empty_cache()
import gc; gc.collect()

results = pd.DataFrame({
    "id": node_ids,
    "anomaly_score": all_scores,
    "predicted_anomaly": (all_scores > optimal_threshold).astype(int),
    "actual_sar": is_sar_all,
})

# Add risk score (0-100 normalized)
results["risk_score"] = (
    (results["anomaly_score"] - results["anomaly_score"].min())
    / (results["anomaly_score"].max() - results["anomaly_score"].min())
    * 100
)

# Merge with node features
results = results.merge(node_features.drop(columns=["is_sar"]), on="id", how="left")

# Correct predictions
results["correct"] = results["predicted_anomaly"] == results["actual_sar"]

print(f"Total nodes:        {len(results):,}")
print(f"Predicted anomaly:  {results['predicted_anomaly'].sum():,}")
print(f"Actual SAR:         {results['actual_sar'].sum():,}")
print(f"Overall accuracy:   {results['correct'].mean():.4f}")
print(f"GPU freed: {torch.cuda.memory_allocated()/1e6:.1f} MB")

Total nodes:        7,500
Predicted anomaly:  3,463
Actual SAR:         644
Overall accuracy:   0.5711
GPU freed: 8.5 MB


## 12. True Positives, False Positives, False Negatives

In [46]:
tp = results[(results["predicted_anomaly"] == 1) & (results["actual_sar"] == 1)]
fp = results[(results["predicted_anomaly"] == 1) & (results["actual_sar"] == 0)]
fn = results[(results["predicted_anomaly"] == 0) & (results["actual_sar"] == 1)]
tn = results[(results["predicted_anomaly"] == 0) & (results["actual_sar"] == 0)]

print(f"True Positives  (SAR caught):     {len(tp):,}")
print(f"False Positives (false alarms):   {len(fp):,}")
print(f"False Negatives (SAR missed):     {len(fn):,}")
print(f"True Negatives  (clean correct):  {len(tn):,}")
print(f"\nDetection rate (recall):  {len(tp) / (len(tp) + len(fn)) * 100:.1f}% of SAR nodes caught")
print(f"False alarm rate:         {len(fp) / (len(fp) + len(tn)) * 100:.1f}% of clean nodes flagged")

True Positives  (SAR caught):     445
False Positives (false alarms):   3,018
False Negatives (SAR missed):     199
True Negatives  (clean correct):  3,838

Detection rate (recall):  69.1% of SAR nodes caught
False alarm rate:         44.0% of clean nodes flagged


## 13. Top Caught SAR Nodes (True Positives)

In [47]:
display_cols = [
    "id", "risk_score", "anomaly_score", "actual_sar",
    "type", "in_degree", "out_degree",
    "total_amount_sent", "total_amount_received",
]

print("Top 15 True Positives (highest risk SAR nodes correctly caught):")
tp.nlargest(15, "anomaly_score")[display_cols]

Top 15 True Positives (highest risk SAR nodes correctly caught):


,id,risk_score,anomaly_score,actual_sar,type,in_degree,out_degree,total_amount_sent,total_amount_received
4960,0ed1750d,38.190113,115.421951,1,0,18.0,1.0,463.61,125017.46
4114,ab35a7fd,21.456182,64.877731,1,0,8.0,0.0,0.00,59297.69
3,c1bfb464,15.146949,45.820923,1,0,30611.0,31934.0,17051866.28,16325916.80
5132,e6ccbd74,15.079130,45.616081,1,0,17.0,2.0,1075.72,121367.74
3239,3499edce,14.074614,42.581978,1,1,20.0,3.0,1309.11,129565.11
3200,b59bd631,13.016316,39.385429,1,1,20.0,1.0,714.29,121547.72
5531,c09d1681,6.694046,20.289249,1,1,3.0,16.0,119784.72,1724.07
5623,e0c98290,5.301051,16.081760,1,1,17.0,2.0,1478.84,68660.89
4908,2c398412,4.000855,12.154570,1,1,4.0,15.0,122819.29,2290.22
1708,10a358d9,3.242980,9.865435,1,0,5.0,18.0,125054.59,2485.64


## 14. Missed SAR Nodes (False Negatives)

In [48]:
print(f"SAR nodes MISSED by the model ({len(fn)} total):")
if len(fn) > 0:
    fn.nlargest(15, "anomaly_score")[display_cols]
else:
    print("None — all SAR nodes detected!")

SAR nodes MISSED by the model (199 total):


## 15. Save Final Predictions

In [ ]:
results.to_parquet(os.path.join(RESULTS_DIR, "predictions.parquet"), index=False)

eval_summary = {
    "roc_auc": float(roc_auc),
    "average_precision": float(ap),
    "optimal_threshold": float(optimal_threshold),
    "accuracy": float(results["correct"].mean()),
    "true_positives": int(len(tp)),
    "false_positives": int(len(fp)),
    "false_negatives": int(len(fn)),
    "true_negatives": int(len(tn)),
    "detection_rate": float(len(tp) / max(len(tp) + len(fn), 1)),
    "false_alarm_rate": float(len(fp) / max(len(fp) + len(tn), 1)),
}
with open(os.path.join(RESULTS_DIR, "eval_summary.json"), "w") as f:
    json.dump(eval_summary, f, indent=2)

print("Saved:")
print(f"  {RESULTS_DIR}/predictions.parquet")
print(f"  {RESULTS_DIR}/eval_summary.json")
print(f"  {RESULTS_DIR}/roc_curve_eval.png")
print(f"  {RESULTS_DIR}/pr_curve_eval.png")
print(f"  {RESULTS_DIR}/score_comparison.png")
print(f"\n{'='*40}")
print(f"FINAL ROC AUC: {roc_auc:.4f}")
print(f"{'='*40}")

In [ ]:
# ── GPU Cleanup — free VRAM for next notebook ──
import gc
for v in ["G", "E", "all_tensor", "train_tensor", "eval_tensor"]:
    if v in dir():
        exec(f"del {v}")
torch.cuda.empty_cache(); gc.collect()
print(f"GPU freed: {torch.cuda.memory_allocated()/1e6:.1f} MB allocated")

GPU freed: 8.5 MB allocated


: 